# LiDAR Patch Processing

This notebook converts a georeferenced LiDAR mosaic into fixed 256 x 256 training patches. Each output GeoTIFF contains band 1 as the LiDAR surface/residual and band 2 as a validity mask. Patch IDs and georeferencing are preserved so Sentinel-1 windows can be matched later.

The patch grid uses a configurable overlap. Existing Tessa patches can be used instead; in that case, skip the extraction cell and point the later notebooks at the existing patch directory.

In [1]:
from pathlib import Path
import json
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.windows import transform as window_transform

## Configuration

Set `MOSAIC_PATH` to the LiDAR mosaic and choose a region. The target patch size is 256 pixels, matching Tessa's model input.

In [2]:
REPO_DIR = Path('/cs/student/project_msc/2025/aibh/jiayiche')
MOSAIC_PATH = Path('/cs/student/project_msc/2025/aibh/jiayiche/raw_data/lidar_data_2024/TukApr16/1m')
REGION = 'tuk'
OUT_DIR = REPO_DIR / 'input_data' / f'lidar_patches_{REGION}'
PATCH_SIZE = 256
OVERLAP = 0.5
MIN_VALID_FRACTION = 0.7   # matches Tessa's valid_threshold, was 0.02
OUT_DIR.mkdir(parents=True, exist_ok=True)



## Locate the mosaic

This accepts either a direct GeoTIFF path or a directory containing one. The source must have a CRS and affine transform.

In [3]:
def resolve_mosaic(path):
    if path.is_file():
        return path
    candidates = sorted(path.glob('*.tif')) + sorted(path.glob('*.tiff'))
    if not candidates:
        raise FileNotFoundError(f'No GeoTIFF found at {path}')
    return candidates[0]

MOSAIC_PATH = resolve_mosaic(MOSAIC_PATH)
with rasterio.open(MOSAIC_PATH) as src:
    print('Mosaic:', MOSAIC_PATH)
    print('Shape:', src.height, src.width, 'bands:', src.count, 'CRS:', src.crs)
    assert src.crs is not None, 'The LiDAR mosaic needs a CRS for Sentinel-1 matching.'


Mosaic: /cs/student/project_msc/2025/aibh/jiayiche/raw_data/lidar_data_2024/TukApr16/1m/warped_2024-04-16.tiff
Shape: 37603 37466 bands: 1 CRS: EPSG:6931


## Extract patches

Only windows with enough finite LiDAR data are written. The mask is finite where the source has usable data, so downstream training can ignore nodata pixels.

In [4]:
def compute_valid_patch_positions(lidar_array, patch_size, stride, valid_threshold):
    H, W = lidar_array.shape
    mask = np.isfinite(lidar_array)
    positions = []
    for y in range(0, H - patch_size + 1, stride):
        for x in range(0, W - patch_size + 1, stride):
            patch_mask = mask[y:y + patch_size, x:x + patch_size]
            if patch_mask.mean() >= valid_threshold:
                positions.append((x, y))
    return positions

def extract_lidar_patches_tessa_method(mosaic_path, output_dir, patch_size, overlap, valid_threshold):
    stride = int(round(patch_size * (1.0 - overlap)))
    with rasterio.open(mosaic_path) as src:
        lidar_array = src.read(1).astype(np.float32)
        lidar_transform = src.transform
        lidar_crs = src.crs
        nodata = src.nodata

    if nodata is not None:
        lidar_array = np.where(lidar_array == nodata, np.nan, lidar_array)
    lidar_array = np.where(np.isfinite(lidar_array), lidar_array, np.nan)

    positions = compute_valid_patch_positions(lidar_array, patch_size, stride, valid_threshold)
    print(f'Valid patch positions: {len(positions)}')

    written = 0
    for x, y in positions:
        lid_patch_raw = lidar_array[y:y + patch_size, x:x + patch_size]
        valid_mask = np.isfinite(lid_patch_raw)
        lid_patch = np.nan_to_num(lid_patch_raw, nan=0.0).astype(np.float32)
        lid_mask = valid_mask.astype(np.float32)
        stacked = np.stack([lid_patch, lid_mask], axis=0)

        window = Window(x, y, patch_size, patch_size)
        profile = {
            'driver': 'GTiff', 'height': patch_size, 'width': patch_size,
            'count': 2, 'dtype': 'float32', 'crs': lidar_crs,
            'transform': window_transform(window, lidar_transform),
            'compress': 'deflate',
        }
        out_path = output_dir / f'lidar_patch_{written:05d}.tif'
        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(stacked)
        written += 1
    return written

n_written = extract_lidar_patches_tessa_method(MOSAIC_PATH, OUT_DIR, PATCH_SIZE, OVERLAP, MIN_VALID_FRACTION)
print('Patches written:', n_written)

Valid patch positions: 1708
Patches written: 1708


## Verify the patch contract

Every file should be 256 x 256 with two bands. The CRS and transform are required by the Sentinel-1 collocation notebook.

In [6]:
patches = sorted(OUT_DIR.glob('lidar_patch_*.tif'))
assert patches, 'No LiDAR patches were produced.'
with rasterio.open(patches[0]) as src:
    assert src.count == 2 and src.shape == (PATCH_SIZE, PATCH_SIZE)
    print('Verified:', patches[0].name, src.shape, src.count, src.crs)

Verified: lidar_patch_00000.tif (256, 256) 2 EPSG:6931
